In [ ]:
!pip install transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 34.6 kB/s  0:03:39m0:00:040:09m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 26.0 kB/s  0:00:32 eta 0:00:05
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 51.2 kB/s  0:01:22 eta 0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 44.7 kB/s  0:01:08 eta 0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 MB 160.5 kB/s  0:10:55m0:00:0100:15
  Attempting uninstall: regex
    Found existing installation: regex 2025.9.1
    Uninstalling regex-2025.9.1:
      Successfully uninstalled regex-2025.9.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [transformers] [transformers]ub]


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

### Load Pretrained model 

#### Over here we're using model_name = "microsoft/DialoGPT-small"
##### we can use "microsoft/DialoGPT-medium" or "microsoft/DialoGPT-large" for better results

In [ ]:
# Load DialoGPT model and tokenizer
model_name = "microsoft/DialoGPT-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set model to evaluation mode
model.eval()

print("Model loaded successfully!")

config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/351M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
# Store chat history
chat_history_ids = None

# Maximum response length
max_length = 1000

print("Chatbot initialized!")

Chatbot initialized!


### Chat Function

In [ ]:
def generate_response(user_input, chat_history_ids, tokenizer, model):
    # Ignore empty input
    if user_input.strip() == "":
        return "Please say something so I can respond.", chat_history_ids

    # Encode input
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    # Create attention mask
    attention_mask = torch.ones(new_input_ids.shape, dtype=torch.long)

    # Append history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
        attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)
    else:
        bot_input_ids = new_input_ids

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.75
    )

    # Decode response
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response, chat_history_ids

In [5]:
# Import necessary libraries
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import ipywidgets as widgets
from IPython.display import display, Markdown

# Load DialoGPT model and tokenizer
model_name = "microsoft/DialoGPT-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set model to evaluation mode
model.eval()

print("Model loaded successfully!")

# Store chat history
chat_history_ids = None

# Maximum response length
max_length = 1000

print("Chatbot initialized!")

def generate_response(user_input, chat_history_ids, tokenizer, model):
    # Ignore empty input
    if user_input.strip() == "":
        return "Please say something so I can respond.", chat_history_ids

    # Encode input
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    # Create attention mask
    attention_mask = torch.ones(new_input_ids.shape, dtype=torch.long)

    # Append history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
        attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)
    else:
        bot_input_ids = new_input_ids

    # Generate response with better parameters
    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=200,  # Reduced max length for more focused responses
        min_length=5,    # Ensure minimum response length
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.9,       # Slightly reduced for more coherent responses
        temperature=0.7,  # Slightly reduced for more predictable responses
        no_repeat_ngram_size=2,  # Prevent repetition
        early_stopping=True
    )

    # Decode response
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response, chat_history_ids

# Create widgets for interactive chat
input_text = widgets.Text(
    value='',
    placeholder='Type your message here...',
    description='User:',
    style={'description_width': 'initial'},
    layout={'width': '500px'}
)

send_button = widgets.Button(
    description='Send',
    button_style='success',
    layout={'width': '100px'}
)

output_area = widgets.Output()

def on_send_click(b):
    global chat_history_ids
    
    user_input = input_text.value
    input_text.value = ''  # Clear input
    
    if user_input.strip() == "":
        with output_area:
            print("Chatbot: Please type something.")
        return
    
    if user_input.lower() in ["exit", "quit"]:
        with output_area:
            print("Chatbot: Goodbye! Have a great day.")
        return
    
    # Display user input
    with output_area:
        display(Markdown(f"**User:** {user_input}"))
    
    # Generate and display response
    response, chat_history_ids = generate_response(user_input, chat_history_ids, tokenizer, model)
    
    with output_area:
        display(Markdown(f"**Chatbot:** {response}"))

# Connect button to function
send_button.on_click(on_send_click)

# Also allow pressing Enter in the text box
input_text.on_submit(lambda x: on_send_click(None))

# Display the interface
display(widgets.HBox([input_text, send_button]))
display(output_area)

print("Chat interface ready! Type your message and click Send or press Enter.")

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

Model loaded successfully!
Chatbot initialized!


/var/folders/t1/wt965jp13hb3gls9w5dcy1bm0000gn/T/ipykernel_3048/1563430436.py:113: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  input_text.on_submit(lambda x: on_send_click(None))


Output()

Chat interface ready! Type your message and click Send or press Enter.
